# 02 · Exploratory data analysis

**Question:** how do the social and economic conditions of a country relate to the performance of the companies that operate there?

This notebook explores the data before any modelling: what the variables look like, how they relate to each other, and how much of firm performance there is to explain at the country level at all.

**Design choices (rules from the build plan):**
- **Outcome:** firm performance as ratios (ROA = net income / total assets, operating margin, revenue growth), since raw amounts are in different currencies. Winsorised at the 1st and 99th percentile.
- **Primary firm sample:** Yahoo Finance companies (consistent across all 34 countries and with sectors). ESEF is used as a cross-check, because the two sources overlap and are not matched.
- **Country conditions ("features"):** two views. *Profile* = average 2019–2023 per country. *Lagged* = the previous year's value (year t−1) for each firm-year, so a firm's results are never explained by information from the same year.
- **Indicators missing for more than 25% of countries are dropped** from the 34-country analysis.
- **n is shown everywhere.** With 34 countries, n is small.

> Everything here is **association, not causation**. P-values are shown for orientation only and are not corrected for the many comparisons made.

**Contents:** 1 Build the analysis tables · 2 Country features · 3 Company outcomes · 4 Country vs firm: what explains performance · 5 Country conditions vs firm performance · 6 Country-level business outcomes · 7 EU vs non-EU · 8 Cross-check with Eurostat · 9 Findings

In [1]:
import pathlib, warnings
import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
con = duckdb.connect(str(ROOT / "data" / "gbc.duckdb"), read_only=True)
q = lambda sql: con.sql(sql).df()

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

PROFILE_YEARS = (2019, 2023)   # window averaged for the country profile
WINSOR = (0.01, 0.99)          # firm ratios are clipped to these percentiles
MIN_FIRMS = 10                 # a country needs this many companies to get a country-level outcome
MAX_MISSING = 0.25             # drop country features missing for more than 25% of countries

dim = q("SELECT * FROM dim_country").set_index("iso3")
names = dim.name
print(f"{len(dim)} countries | {PROFILE_YEARS[0]}–{PROFILE_YEARS[1]} profile window | winsorise {WINSOR} | min firms {MIN_FIRMS}")

34 countries | 2019–2023 profile window | winsorise (0.01, 0.99) | min firms 10


## 1 · Build the analysis tables
### 1.1 Country features
36 candidate features from the World Bank, WGI and IMF (the three sources covering all 34 countries). Where several sources measure the same thing, one is kept: World Bank growth, inflation and unemployment (IMF versions dropped), and GDP per person in PPP terms. Size variables (GDP, population, patents) are log-transformed because they are extremely skewed.

Three indicators are kept apart as **outcomes** instead of features: new business density, stock-market value as % of GDP, and the number of listed companies.

In [2]:
# (short name, indicator_id, transform, group)
FEATURES = [
 ("gdp_per_capita_ppp", "WB:NY.GDP.PCAP.PP.KD", "log", "economy"), ("gdp_usd", "WB:NY.GDP.MKTP.CD", "log", "economy"),
 ("gdp_growth", "WB:NY.GDP.MKTP.KD.ZG", None, "economy"), ("inflation", "WB:FP.CPI.TOTL.ZG", None, "economy"),
 ("manufacturing_share", "WB:NV.IND.MANF.ZS", None, "economy"), ("services_share", "WB:NV.SRV.TOTL.ZS", None, "economy"),
 ("trade_gdp", "WB:NE.TRD.GNFS.ZS", None, "trade"), ("fdi_gdp", "WB:BX.KLT.DINV.WD.GD.ZS", None, "trade"),
 ("hightech_exports", "WB:TX.VAL.TECH.MF.ZS", None, "trade"),
 ("private_credit_gdp", "WB:FS.AST.PRVT.GD.ZS", None, "finance"), ("lending_rate", "WB:FR.INR.LEND", None, "finance"),
 ("tax_revenue_gdp", "WB:GC.TAX.TOTL.GD.ZS", None, "fiscal"), ("gov_debt_gdp", "IMF:GGXWDG_NGDP", None, "fiscal"),
 ("current_account_gdp", "IMF:BCA_NGDPD", None, "economy"),
 ("unemployment", "WB:SL.UEM.TOTL.ZS", None, "labour"), ("youth_unemployment", "WB:SL.UEM.1524.ZS", None, "labour"),
 ("labour_participation", "WB:SL.TLF.CACT.ZS", None, "labour"), ("self_employed", "WB:SL.EMP.SELF.ZS", None, "labour"),
 ("population", "WB:SP.POP.TOTL", "log", "social"), ("pop_working_age", "WB:SP.POP.1564.TO.ZS", None, "social"),
 ("pop_65plus", "WB:SP.POP.65UP.TO.ZS", None, "social"), ("urban_share", "WB:SP.URB.TOTL.IN.ZS", None, "social"),
 ("life_expectancy", "WB:SP.DYN.LE00.IN", None, "social"), ("gini", "WB:SI.POV.GINI", None, "social"),
 ("tertiary_enrolment", "WB:SE.TER.ENRR", None, "education"), ("education_spend_gdp", "WB:SE.XPD.TOTL.GD.ZS", None, "education"),
 ("rd_gdp", "WB:GB.XPD.RSDV.GD.ZS", None, "innovation"), ("patents_residents", "WB:IP.PAT.RESD", "log", "innovation"),
 ("internet_users", "WB:IT.NET.USER.ZS", None, "infrastructure"), ("mobile_subscriptions", "WB:IT.CEL.SETS.P2", None, "infrastructure"),
 ("gov_effectiveness", "WGI:GOV_WGI_GE.EST", None, "governance"), ("regulatory_quality", "WGI:GOV_WGI_RQ.EST", None, "governance"),
 ("rule_of_law", "WGI:GOV_WGI_RL.EST", None, "governance"), ("corruption_control", "WGI:GOV_WGI_CC.EST", None, "governance"),
 ("voice_accountability", "WGI:GOV_WGI_VA.EST", None, "governance"), ("political_stability", "WGI:GOV_WGI_PV.EST", None, "governance"),
]
OUTCOMES_A = [("new_business_density", "WB:IC.BUS.NDNS.ZS", None), ("market_cap_gdp", "WB:CM.MKT.LCAP.GD.ZS", None),
              ("listed_companies", "WB:CM.MKT.LDOM.NO", "log")]
spec = {s: (i, t) for s, i, t, g in FEATURES} | {s: (i, t) for s, i, t in OUTCOMES_A}
group = {s: g for s, i, t, g in FEATURES}

fc = q("SELECT iso3, indicator_id, year, value FROM fact_country_year WHERE indicator_id IN (" + ",".join(f"'{i}'" for i, t in spec.values()) + ")")
id2short = {i: s for s, (i, t) in spec.items()}
panel = fc.assign(short=fc.indicator_id.map(id2short)).pivot_table(index=["iso3", "year"], columns="short", values="value")
panel.columns.name = None
for s, (i, t) in spec.items():
    if t == "log":
        panel[s] = np.log(panel[s].where(panel[s] > 0))
panel = panel.reset_index()
print("country-year panel:", panel.shape, "| years", panel.year.min(), "-", panel.year.max())

# profile = mean over the window, needing at least 3 years of data
win = panel[panel.year.between(*PROFILE_YEARS)]
profile_all = win.groupby("iso3")[list(spec)].agg(lambda s: s.mean() if s.notna().sum() >= 3 else np.nan)
cov = profile_all.notna().mean().rename("share_of_countries_with_value")
dropped = cov[(cov < 1 - MAX_MISSING) & cov.index.isin(group)]
feat_cols = [s for s in group if s not in dropped.index]
profile = profile_all[feat_cols]
print(f"features kept: {len(feat_cols)} of {len(group)} | dropped for >{MAX_MISSING:.0%} missing: {list(dropped.index)}")
cov.to_frame().assign(kept=lambda d: d.index.isin(feat_cols), group=lambda d: d.index.map(group).fillna("outcome (level A)")).sort_values("share_of_countries_with_value").head(12)

country-year panel: (442, 41) | years 2012 - 2024
features kept: 35 of 36 | dropped for >25% missing: ['lending_rate']


,share_of_countries_with_value,kept,group
lending_rate,0.382,False,finance
market_cap_gdp,0.647,False,outcome (level A)
listed_companies,0.647,False,outcome (level A)
gini,0.765,True,social
tax_revenue_gdp,0.912,True,fiscal
rd_gdp,0.912,True,innovation
education_spend_gdp,0.941,True,education
private_credit_gdp,0.941,True,finance
new_business_density,0.971,False,outcome (level A)
life_expectancy,1.000,True,social


### 1.2 Company-years
One row per company and fiscal year, from `v_company_year`. The Yahoo sample is the primary one. Ratios are winsorised; **sector-adjusted ROA** subtracts the median ROA of the company's sector, so that a country with many banks or utilities is not compared with one full of tech firms.

In [3]:
cy_all = q("SELECT * FROM v_company_year")
yf = cy_all[(cy_all.source_id == "YF") & cy_all.roa.notna()].copy()
for col in ["roa", "operating_margin", "revenue_growth"]:
    lo, hi = yf[col].quantile(WINSOR)
    yf[col + "_w"] = yf[col].clip(lo, hi)
yf["sector"] = yf.sector.fillna("Unknown")
yf["roa_sector_adj"] = yf.roa_w - yf.groupby("sector").roa_w.transform("median")
yf["country"] = yf.iso3.map(names)

# country features of the PREVIOUS year (year t-1; if missing, t-2) for each firm-year
grid = pd.MultiIndex.from_product([panel.iso3.unique(), range(2010, 2027)], names=["iso3", "year"])
full = panel.set_index(["iso3", "year"]).reindex(grid)[feat_cols + [o[0] for o in OUTCOMES_A]]
lag = full.groupby(level="iso3").ffill(limit=1).groupby(level="iso3").shift(1)
fy = yf.merge(lag[feat_cols].reset_index().rename(columns={"year": "fiscal_year"}), on=["iso3", "fiscal_year"], how="left")
print(f"Yahoo company-years: {len(fy):,} | companies: {fy.company_id.nunique():,} | countries: {fy.iso3.nunique()} | fiscal years {fy.fiscal_year.min()}–{fy.fiscal_year.max()}")
print(f"Company-years with all lagged features present: {fy[feat_cols].notna().all(axis=1).mean():.0%}")
cov_lag = fy[feat_cols].notna().mean().sort_values()
print("Lowest lagged-feature coverage:", {k: round(v, 2) for k, v in cov_lag.head(4).items()})
fy[["name", "country", "sector", "fiscal_year", "roa", "roa_w", "roa_sector_adj", "operating_margin_w", "revenue_growth_w"]].sample(6, random_state=1)

Yahoo company-years: 10,220 | companies: 2,590 | countries: 34 | fiscal years 2021–2026
Company-years with all lagged features present: 36%
Lowest lagged-feature coverage: {'patents_residents': 0.51, 'education_spend_gdp': 0.74, 'gini': 0.8, 'tax_revenue_gdp': 0.89}


,name,country,sector,fiscal_year,roa,roa_w,roa_sector_adj,operating_margin_w,revenue_growth_w
5633,Investor AB (publ),Sweden,Financial Services,2025,0.143,0.143,0.131,NaN,0.282
5769,Jazz Pharmaceuticals plc,Ireland,Healthcare,2022,-0.021,-0.021,-0.065,0.140,NaN
2625,Booking Holdings Inc.,United States,Consumer Cyclical,2024,0.212,0.212,0.163,0.318,0.111
2615,BioPorto A/S,Denmark,Healthcare,2022,-0.699,-0.398,-0.442,-2.725,NaN
6434,"Mondelez International, Inc.",United States,Consumer Defensive,2022,0.038,0.038,-0.014,0.121,NaN
1925,ams-OSRAM AG,Austria,Technology,2023,-0.218,-0.218,-0.277,-0.036,-0.255


## 2 · Country features
### 2.1 Distributions
Each panel is the 2019–2023 average across the 34 countries. Strong skew or clusters tell us where transformations or ranks (Spearman) are safer than raw values.

In [4]:
cols = feat_cols
n_cols = 6
n_rows = int(np.ceil(len(cols) / n_cols))
fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=cols, vertical_spacing=0.06)
for k, c in enumerate(cols):
    fig.add_trace(go.Histogram(x=profile[c].dropna(), nbinsx=12, marker_color="#4C78A8", showlegend=False), row=k // n_cols + 1, col=k % n_cols + 1)
fig.update_layout(height=170 * n_rows, margin=dict(t=40, b=20), title="Distribution of each country feature (34 countries)")
fig.update_annotations(font_size=10)
fig.show()
skew = profile.skew().rename("skewness").to_frame().assign(missing=profile.isna().sum())
print("Most skewed features (|skew| > 1):")
skew[skew.skewness.abs() > 1].sort_values("skewness")

Most skewed features (|skew| > 1):


,skewness,missing
internet_users,-1.831,0
voice_accountability,-1.611,0
life_expectancy,-1.291,0
manufacturing_share,1.007,0
gov_debt_gdp,1.351,0
current_account_gdp,1.445,0
gdp_growth,1.608,0
gini,1.916,8
hightech_exports,1.968,0
trade_gdp,2.036,0


### 2.2 How the country features relate to each other
Spearman correlation (ranks, so robust to outliers). Features are ordered by hierarchical clustering so related variables sit together. Large blocks of strong colour mean the features carry **overlapping information**: many "different" indicators are really one underlying factor (development level).

In [5]:
corr = profile.corr(method="spearman")
dist = squareform(1 - corr.abs().fillna(0).values, checks=False)
order = hierarchy.leaves_list(hierarchy.linkage(dist, "average"))
co = corr.iloc[order, order]
fig = px.imshow(co, color_continuous_scale="RdBu", zmin=-1, zmax=1, aspect="auto", title="Spearman correlation between country features (2019–2023 profile)")
fig.update_layout(height=820, xaxis_tickangle=-60)
fig.show()
pairs = corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack().rename("rho").reset_index()
pairs.columns = ["feature_a", "feature_b", "rho"]
strong = pairs[pairs.rho.abs() >= 0.85].sort_values("rho", key=abs, ascending=False)
print(f"{len(strong)} feature pairs with |rho| >= 0.85 (of {len(pairs)}). Strongest:")
strong.head(15)

15 feature pairs with |rho| >= 0.85 (of 1225). Strongest:


,feature_a,feature_b,rho
1082,regulatory_quality,corruption_control,0.968
1117,rule_of_law,corruption_control,0.950
1047,gov_effectiveness,corruption_control,0.940
1045,gov_effectiveness,regulatory_quality,0.939
1081,regulatory_quality,rule_of_law,0.936
1046,gov_effectiveness,rule_of_law,0.932
61,gdp_usd,patents_residents,0.904
469,unemployment,youth_unemployment,0.896
30,gdp_per_capita_ppp,regulatory_quality,0.885
1118,rule_of_law,voice_accountability,0.882


### 2.3 Is there a hidden structure? Principal components
PCA on the standardised features (missing values replaced by the mean, which keeps those countries neutral on that feature). If one or two components explain most of the variance, the countries line up along a small number of dimensions, usually *development level* first.

In [6]:
z = (profile - profile.mean()) / profile.std()
Z = z.fillna(0).values
U, S, Vt = np.linalg.svd(Z, full_matrices=False)
expl = S**2 / (S**2).sum()
scores = pd.DataFrame(U[:, :3] * S[:3], index=profile.index, columns=["PC1", "PC2", "PC3"])
loadings = pd.DataFrame(Vt[:3].T, index=profile.columns, columns=["PC1", "PC2", "PC3"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], subplot_titles=("Variance explained", "Countries on the first two components"))
fig.add_trace(go.Bar(x=[f"PC{i+1}" for i in range(8)], y=expl[:8] * 100, marker_color="#4C78A8", showlegend=False), row=1, col=1)
for reg, g in scores.join(dim[["region", "name"]]).groupby("region"):
    fig.add_trace(go.Scatter(x=g.PC1, y=g.PC2, mode="markers+text", text=g.index, textposition="top center", name=reg, marker=dict(size=9)), row=1, col=2)
fig.update_xaxes(title_text="PC1", row=1, col=2); fig.update_yaxes(title_text="PC2", row=1, col=2); fig.update_yaxes(title_text="% of variance", row=1, col=1)
fig.update_layout(height=520)
fig.show()
print(f"PC1 explains {expl[0]:.0%}, PC2 {expl[1]:.0%}, PC3 {expl[2]:.0%} (first three: {expl[:3].sum():.0%})")
top_load = {f"PC{i+1}": loadings[f"PC{i+1}"].reindex(loadings[f"PC{i+1}"].abs().sort_values(ascending=False).index).head(6).round(2) for i in (0, 1)}
pc1_desc = ", ".join(f"{f} ({v:+.2f})" for f, v in list(top_load["PC1"].items())[:4])
print("Features that load most strongly (positive or negative):")
pd.concat({k: v.rename("loading").reset_index().rename(columns={"index": "feature"}) for k, v in top_load.items()}, axis=1)

PC1 explains 34%, PC2 15%, PC3 12% (first three: 60%)
Features that load most strongly (positive or negative):


PC1                        PC2        
              feature loading            feature loading
0  regulatory_quality  -0.280  patents_residents   0.360
1  corruption_control  -0.280            gdp_usd   0.350
2         rule_of_law  -0.280         population   0.270
3   gov_effectiveness  -0.270   hightech_exports   0.260
4  gdp_per_capita_ppp  -0.270          inflation  -0.250
5      internet_users  -0.260             rd_gdp   0.240

## 3 · Company outcomes
### 3.1 What do the outcome ratios look like?
Raw ratios have extreme outliers (a company with tiny assets can show a ROA of several hundred per cent), which is why the 1st and 99th percentiles are used for winsorising. The table shows how much winsorising changes the picture.

In [7]:
rows = []
for c in ["roa", "operating_margin", "revenue_growth"]:
    s, w = yf[c].dropna(), yf[c + "_w"].dropna()
    rows.append({"ratio": c, "n": len(s), "raw min": s.min(), "raw max": s.max(), "1st pct": s.quantile(.01), "median": s.median(),
                 "99th pct": s.quantile(.99), "mean (raw)": s.mean(), "mean (winsorised)": w.mean(), "share missing": 1 - len(s) / len(yf)})
pd.DataFrame(rows).set_index("ratio")

,n,raw min,raw max,1st pct,median,99th pct,mean (raw),mean (winsorised),share missing
ratio,,,,,,,,,
roa,10220,-719.202,97.853,-0.398,0.038,0.310,-0.114,0.041,0.000
operating_margin,9131,"-1,380.701",127.333,-2.928,0.116,0.791,-0.575,0.116,0.107
revenue_growth,7493,-23.357,"3,010.587",-0.723,0.054,1.975,0.793,0.088,0.267


In [8]:
fig = make_subplots(rows=1, cols=3, subplot_titles=("ROA", "Operating margin", "Revenue growth"))
for k, c in enumerate(["roa_w", "operating_margin_w", "revenue_growth_w"], 1):
    fig.add_trace(go.Histogram(x=yf[c].dropna(), nbinsx=60, marker_color="#4C78A8", showlegend=False), row=1, col=k)
    fig.add_vline(x=0, line_color="grey", row=1, col=k)
fig.update_layout(height=360, title="Winsorised company outcomes (Yahoo company-years)")
fig.show()

### 3.2 Over time
Median outcomes by fiscal year, for both sources. The year-to-year swings are a reminder that macro conditions (2022 energy shock, inflation, rate rises) affect all countries at once. If ROA changes a lot from year to year, comparing countries over different years can be misleading.

In [9]:
cy_w = cy_all[cy_all.roa.notna()].copy()
cy_w["roa_w"] = cy_w.roa.clip(*cy_w.roa.quantile(WINSOR))
t = cy_w.groupby(["fiscal_year", "source_id"]).agg(median_roa=("roa_w", "median"), firm_years=("roa_w", "size")).reset_index()
t = t[t.firm_years >= 100]
fig = px.line(t, x="fiscal_year", y="median_roa", color="source_id", markers=True, title="Median ROA by fiscal year (years with at least 100 company-years)")
fig.update_layout(height=380)
fig.show()
t.pivot(index="fiscal_year", columns="source_id", values="firm_years")

source_id,ESEF,YF
fiscal_year,,
2020,626.000,NaN
2021,930.000,NaN
2022,"1,077.000","2,528.000"
2023,"1,077.000","2,547.000"
2024,970.000,"2,547.000"
2025,562.000,"2,507.000"


### 3.3 Do the two sources agree?
For the EU countries covered by both, ESEF and Yahoo should give similar country medians if the data are sound. Differences come from different companies (ESEF includes many smaller firms), different accounting standards, and different fiscal years.

In [10]:
both = cy_w[cy_w.iso3.isin(cy_w[cy_w.source_id == "ESEF"].iso3.unique())]
cm = both.groupby(["iso3", "source_id"]).agg(median_roa=("roa_w", "median"), companies=("company_id", "nunique")).reset_index()
wide = cm.pivot(index="iso3", columns="source_id", values="median_roa").dropna()
r, p = stats.spearmanr(wide.YF, wide.ESEF)
agree_r = r
print(f"Countries covered by both sources: {len(wide)}; Spearman ρ between their median ROA = {r:.2f} (p = {p:.3f})")
fig = px.scatter(wide.reset_index(), x="YF", y="ESEF", text="iso3", title=f"Country median ROA: Yahoo vs ESEF (n={len(wide)} countries, Spearman ρ = {r:.2f})")
lim = [min(wide.min()) - .01, max(wide.max()) + .01]
fig.add_trace(go.Scatter(x=lim, y=lim, mode="lines", line=dict(dash="dot", color="grey"), name="equal"))
fig.update_traces(textposition="top center", selector=dict(mode="markers+text")); fig.update_layout(height=480)
fig.show()

Countries covered by both sources: 19; Spearman ρ between their median ROA = 0.46 (p = 0.048)


## 4 · How much of firm performance is about the country at all?
The plan's own warning: *company performance depends mostly on the firm and its sector; country effects exist but are smaller.* Here is a direct test. For each grouping (country, sector, company) we measure the share of ROA variance explained by group differences (R²) and the **adjusted** R² (which penalises groupings with many small groups, such as 2,500 companies with about 4 years each).

In [11]:
d = fy.dropna(subset=["roa_w"]).copy()
y = d.roa_w.values
def r2_groups(keys):
    X = pd.get_dummies(d[keys].astype(str), drop_first=True).astype(float) if isinstance(keys, list) else pd.get_dummies(d[keys].astype(str), drop_first=True).astype(float)
    X = np.column_stack([np.ones(len(d)), X.values])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    res = y - X @ beta
    r2 = 1 - (res**2).sum() / ((y - y.mean())**2).sum()
    k = X.shape[1] - 1
    return r2, 1 - (1 - r2) * (len(y) - 1) / (len(y) - k - 1), k + 1
out = []
for label, keys in [("country", "iso3"), ("sector", "sector"), ("fiscal year", "fiscal_year"), ("country + sector", ["iso3", "sector"]),
                    ("country + sector + fiscal year", ["iso3", "sector", "fiscal_year"]), ("company (firm identity)", "company_id")]:
    r2, adj, g = r2_groups(keys)
    out.append({"grouping": label, "groups": g, "R²": r2, "adjusted R²": adj})
var = pd.DataFrame(out).set_index("grouping")
inc = var.loc["country + sector", "adjusted R²"] - var.loc["sector", "adjusted R²"]
print(f"n = {len(d):,} company-years. Adding country to sector raises adjusted R² by {inc:.3f}.")
fig = px.bar(var.reset_index(), x="adjusted R²", y="grouping", orientation="h", text=var["adjusted R²"].round(3).values,
             title="Share of ROA variance explained (adjusted R²)")
fig.update_layout(height=380, yaxis_autorange="reversed")
fig.show()
var

n = 10,220 company-years. Adding country to sector raises adjusted R² by 0.042.


,groups,R²,adjusted R²
grouping,,,
country,34,0.044,0.041
sector,12,0.026,0.025
fiscal year,6,0.001,0.000
country + sector,45,0.071,0.067
country + sector + fiscal year,50,0.072,0.067
company (firm identity),2590,0.728,0.635


**How to read it:** an adjusted R² near 0 means the grouping says nothing about ROA; near 1 means it explains nearly everything. Firm identity can score high simply because a company's profitability is persistent from year to year, while the country share is what the research question can hope to explain.

Country medians (next section) remove firm noise by averaging, so they can show clearer patterns across countries than the firm-year data.

## 5 · Country conditions vs firm performance
Three views of the same question, from coarse to fine:

1. **Country level:** median ROA of each country's firms vs the country's 2019–2023 profile. *n is about 34.*
2. **Firm-year level:** each company-year's ROA vs the previous year's country feature. *n is about 10,000, but firms in the same country share one feature value, so the effective sample is closer to the number of countries.*
3. **Sector-adjusted firm-year:** same as 2, after subtracting the sector median, to remove differences caused by the sector mix.

Spearman ρ (rank correlation). P-values are not corrected for multiple testing: with 35 features, a few will look "significant" by chance.

In [12]:
cn = (fy.groupby("iso3").agg(median_roa=("roa_w", "median"), median_margin=("operating_margin_w", "median"), median_growth=("revenue_growth_w", "median"),
                             median_roa_adj=("roa_sector_adj", "median"), firms=("company_id", "nunique"), firm_years=("roa_w", "size")))
cn = cn[cn.firms >= MIN_FIRMS]
print(f"countries with at least {MIN_FIRMS} companies: {len(cn)}")

def spear(x, y):
    m = x.notna() & y.notna()
    if m.sum() < 8:
        return np.nan, np.nan, int(m.sum())
    r, p = stats.spearmanr(x[m], y[m])
    return r, p, int(m.sum())

res = []
for f in feat_cols:
    r1, p1, n1 = spear(profile.loc[cn.index, f], cn.median_roa)
    r2, p2, n2 = spear(fy[f], fy.roa_w)
    r3, p3, n3 = spear(fy[f], fy.roa_sector_adj)
    res.append({"feature": f, "group": group[f], "ρ country median ROA": r1, "p": p1, "n countries": n1,
                "ρ firm-year ROA": r2, "ρ sector-adjusted": r3, "n firm-years": n2})
res = pd.DataFrame(res).set_index("feature")
res.sort_values("ρ country median ROA", key=abs, ascending=False).head(15)

countries with at least 10 companies: 34


,group,ρ country median ROA,p,n countries,ρ firm-year ROA,ρ sector-adjusted,n firm-years
feature,,,,,,,
internet_users,infrastructure,-0.434,0.010,34,-0.115,-0.129,10220
trade_gdp,trade,-0.404,0.018,34,-0.128,-0.136,10220
life_expectancy,social,-0.389,0.023,34,-0.115,-0.135,10220
urban_share,social,-0.355,0.039,34,-0.096,-0.113,10220
pop_65plus,social,-0.329,0.058,34,-0.101,-0.108,10220
gdp_per_capita_ppp,economy,-0.327,0.059,34,-0.098,-0.112,10220
rule_of_law,governance,-0.319,0.066,34,-0.097,-0.112,10220
gov_effectiveness,governance,-0.318,0.067,34,-0.089,-0.107,10220
population,social,0.318,0.067,34,0.119,0.129,10220


In [13]:
rs = res.sort_values("ρ country median ROA")
fig = go.Figure()
fig.add_trace(go.Bar(y=rs.index, x=rs["ρ country median ROA"], orientation="h", name="country median ROA (n≈34)", marker_color="#4C78A8"))
fig.add_trace(go.Bar(y=rs.index, x=rs["ρ sector-adjusted"], orientation="h", name="firm-year, sector-adjusted", marker_color="#F58518"))
fig.update_layout(barmode="group", height=900, title="Spearman ρ of each country feature with firm performance", xaxis_title="ρ", legend=dict(orientation="h", y=1.04))
fig.show()

**The strongest country-level relationships, as scatter plots** (each point is a country; the line is a simple least-squares fit; the label gives the country code).

In [14]:
top = res["ρ country median ROA"].abs().sort_values(ascending=False).head(6).index
fig = make_subplots(rows=2, cols=3, subplot_titles=[f"{f} (ρ={res.loc[f, 'ρ country median ROA']:.2f})" for f in top], vertical_spacing=0.14)
for k, f in enumerate(top):
    df = pd.DataFrame({"x": profile.loc[cn.index, f], "y": cn.median_roa}).dropna()
    r, c = k // 3 + 1, k % 3 + 1
    fig.add_trace(go.Scatter(x=df.x, y=df.y, mode="markers+text", text=df.index, textposition="top center", textfont_size=8, marker=dict(size=7, color="#4C78A8"), showlegend=False), row=r, col=c)
    b, a = np.polyfit(df.x, df.y, 1)
    xs = np.array([df.x.min(), df.x.max()])
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False), row=r, col=c)
fig.update_layout(height=700, title="Median ROA of a country's companies vs country feature")
fig.update_yaxes(title_text="median ROA")
fig.show()

### 5.1 Do the relationships hold when we change the question?
A pattern worth trusting should show up in more than one view. The table lists features whose sign agrees in all three views. **Caveat:** the features are strongly correlated with each other (section 2.2), so agreement is largely one underlying signal counted many times, not 35 independent confirmations.

In [15]:
chk = res[["ρ country median ROA", "ρ firm-year ROA", "ρ sector-adjusted"]].copy()
chk["strongest |ρ|"] = chk.abs().max(axis=1)
chk["same sign in all 3"] = np.sign(chk[["ρ country median ROA", "ρ firm-year ROA", "ρ sector-adjusted"]]).nunique(axis=1) == 1
print(f"{int(chk['same sign in all 3'].sum())} of {len(chk)} features have the same sign in all three views.")
chk[chk["same sign in all 3"]].sort_values("strongest |ρ|", ascending=False).head(12)

33 of 35 features have the same sign in all three views.


,ρ country median ROA,ρ firm-year ROA,ρ sector-adjusted,strongest |ρ|,same sign in all 3
feature,,,,,
internet_users,-0.434,-0.115,-0.129,0.434,True
trade_gdp,-0.404,-0.128,-0.136,0.404,True
life_expectancy,-0.389,-0.115,-0.135,0.389,True
urban_share,-0.355,-0.096,-0.113,0.355,True
pop_65plus,-0.329,-0.101,-0.108,0.329,True
gdp_per_capita_ppp,-0.327,-0.098,-0.112,0.327,True
rule_of_law,-0.319,-0.097,-0.112,0.319,True
gov_effectiveness,-0.318,-0.089,-0.107,0.318,True
population,0.318,0.119,0.129,0.318,True


### 5.2 Other outcome measures at country level
ROA is one measure. Operating margin and revenue growth tell different stories (growth especially depends on the year and on inflation). The heatmap shows the strongest country-feature relationships for each outcome.

In [16]:
outs = {"median ROA": "median_roa", "sector-adj ROA": "median_roa_adj", "median op. margin": "median_margin", "median revenue growth": "median_growth"}
mat = pd.DataFrame({lab: {f: spear(profile.loc[cn.index, f], cn[c])[0] for f in feat_cols} for lab, c in outs.items()})
mat = mat.loc[mat.abs().max(axis=1).sort_values(ascending=False).index]
fig = px.imshow(mat.head(25), color_continuous_scale="RdBu", zmin=-.7, zmax=.7, aspect="auto", text_auto=".2f",
                title="Spearman ρ: country features (rows) vs four firm outcomes (25 strongest features)")
fig.update_layout(height=760)
fig.show()

## 6 · Country-level business outcomes (no firm data needed)
Three indicators describe the business environment directly: **new business density** (new companies per 1,000 people), **stock-market value as % of GDP**, and the **number of listed companies**. They cover more countries and years than the firm sample and are an independent view of the same question.

In [17]:
outA = [o[0] for o in OUTCOMES_A]
a = pd.DataFrame({o: {f: spear(profile_all.loc[:, f], profile_all[o])[0] for f in feat_cols} for o in outA})
print("countries with a value:", profile_all[outA].notna().sum().to_dict())
a = a.loc[a.abs().max(axis=1).sort_values(ascending=False).index]
fig = px.imshow(a.head(22), color_continuous_scale="RdBu", zmin=-.9, zmax=.9, aspect="auto", text_auto=".2f",
                title="Spearman ρ: country features vs business-environment outcomes (22 strongest features)")
fig.update_layout(height=700)
fig.show()
for o in outA:
    top3 = a[o].dropna().reindex(a[o].abs().dropna().sort_values(ascending=False).index).head(3)
    print(f"{o} (n={int(profile_all[o].notna().sum())} countries): " + ", ".join(f"{f} ρ={v:+.2f}" for f, v in top3.items()))

countries with a value: {'new_business_density': 33, 'market_cap_gdp': 22, 'listed_companies': 22}


new_business_density (n=33 countries): population ρ=-0.44, gdp_usd ρ=-0.38, corruption_control ρ=+0.37
market_cap_gdp (n=22 countries): private_credit_gdp ρ=+0.77, trade_gdp ρ=-0.50, gdp_usd ρ=+0.50
listed_companies (n=22 countries): gdp_usd ρ=+0.85, patents_residents ρ=+0.81, trade_gdp ρ=-0.70


In [18]:
fig = px.scatter(profile_all.join(dim[["name", "region"]]).reset_index(), x="rule_of_law", y="new_business_density", color="region", text="iso3",
                 title="New business density vs rule of law (2019–2023 averages)")
fig.update_traces(textposition="top center", marker_size=9); fig.update_layout(height=520)
fig.show()

## 7 · EU vs non-EU, and groups within the EU
Do firms in EU countries perform differently from those elsewhere? Tests are done on **country medians** (so that big markets don't dominate). Mann–Whitney (two groups) and Kruskal–Wallis (several groups) compare ranks.

In [19]:
cn2 = cn.join(dim[["is_eu", "eu_group", "region", "income_group", "name"]])
cn2["group"] = np.where(cn2.is_eu, "EU", "non-EU")
u = stats.mannwhitneyu(cn2[cn2.is_eu].median_roa, cn2[~cn2.is_eu].median_roa)
print(f"EU: n={cn2.is_eu.sum()}, median of country medians {cn2[cn2.is_eu].median_roa.median():.3f} | non-EU: n={(~cn2.is_eu).sum()}, {cn2[~cn2.is_eu].median_roa.median():.3f} | Mann–Whitney p = {u.pvalue:.3f}")
fig = make_subplots(rows=1, cols=2, subplot_titles=("EU vs non-EU", "EU groups"))
for g, df in cn2.groupby("group"):
    fig.add_trace(go.Box(y=df.median_roa, name=f"{g} (n={len(df)})", boxpoints="all", text=df.name, jitter=.4), row=1, col=1)
eu = cn2[cn2.is_eu]
for g, df in eu.groupby("eu_group"):
    fig.add_trace(go.Box(y=df.median_roa, name=f"{g} (n={len(df)})", boxpoints="all", text=df.name, jitter=.4), row=1, col=2)
fig.update_layout(height=450, showlegend=False, title="Country median ROA")
fig.show()
kw = stats.kruskal(*[g.median_roa.values for _, g in eu.groupby("eu_group")])
print(f"Kruskal–Wallis across EU groups: p = {kw.pvalue:.3f}")
cn2.groupby("income_group").median_roa.agg(["count", "median"]).rename(columns={"count": "countries", "median": "median of country medians"})

EU: n=21, median of country medians 0.032 | non-EU: n=13, 0.043 | Mann–Whitney p = 0.040


Kruskal–Wallis across EU groups: p = 0.524


,countries,median of country medians
income_group,,
High income,28,0.033
Lower middle income,1,0.069
Upper middle income,5,0.046


## 8 · Cross-check with Eurostat's corporate profitability (EU only)
Eurostat publishes economy-wide return on equity and profit share for **all** non-financial corporations. If Yahoo's listed-company ROA picks up something real about country conditions, the two should at least loosely agree. A weak link is expected: listed firms are a small, large-company slice, and ROA and ROE differ.

In [20]:
est = q("""
SELECT f.iso3, i.filters.na_item AS item, AVG(f.value) AS value
FROM fact_country_year f JOIN dim_indicator i USING (indicator_id)
WHERE i.source_code = 'nasa_10_ki' AND f.year BETWEEN 2019 AND 2023 GROUP BY 1, 2
""").pivot(index="iso3", columns="item", values="value")
x = cn.join(est, how="inner")
rows = []
for item in [c for c in est.columns if c in ("ROE_S11", "B2G_B3G_RAT_S11", "IRG_S11", "ROCE_S11")]:
    r, p, n = spear(x.median_roa, x[item])
    rows.append({"Eurostat measure": item, "ρ with Yahoo median ROA": r, "p": p, "n countries": n})
eu_chk = pd.DataFrame(rows).set_index("Eurostat measure")
eu_chk

,ρ with Yahoo median ROA,p,n countries
Eurostat measure,,,
B2G_B3G_RAT_S11,0.194,0.401,21
IRG_S11,-0.119,0.606,21
ROCE_S11,0.143,0.570,18
ROE_S11,0.018,0.938,21


In [21]:
x2 = x.reset_index()
fig = px.scatter(x2, x="ROE_S11", y="median_roa", text="iso3", title="Yahoo median ROA vs Eurostat return on equity of all non-financial corporations (EU, 2019–2023)")
fig.update_traces(textposition="top center", marker_size=9); fig.update_layout(height=480)
fig.show()

## 9 · Findings so far and next steps
The numbers below are computed from the data above when the notebook runs.

In [22]:
top10 = res["ρ country median ROA"].abs().sort_values(ascending=False).head(10)
n_neg = int((res.loc[top10.index, "ρ country median ROA"] < 0).sum())
n_sig = int((res.p < 0.05).sum())
lines = [
 f"1. Sample: {fy.company_id.nunique():,} Yahoo companies in {fy.iso3.nunique()} countries ({len(fy):,} company-years). Every country has at least {MIN_FIRMS} companies.",
 f"2. Country features: the first principal component explains {expl[0]:.0%} of their variance (main drivers: {pc1_desc}); {len(strong)} of {len(pairs)} feature pairs have |ρ| ≥ 0.85, mostly among the governance indicators, GDP per person and the size/demography variables. So they overlap, but the {len(feat_cols)} indicators are far from one single factor.",
 f"3. Variance in firm-year ROA, adjusted R²: company identity {var.loc['company (firm identity)','adjusted R²']:.3f}, country {var.loc['country','adjusted R²']:.3f}, sector {var.loc['sector','adjusted R²']:.3f}, fiscal year {var.loc['fiscal year','adjusted R²']:.3f}; country + sector {var.loc['country + sector','adjusted R²']:.3f}. Firm identity dominates; country conditions can explain only a few per cent.",
 f"4. Country level (n={len(cn)}): the strongest associations with median ROA are " + ", ".join(f"{f} (ρ={res.loc[f,'ρ country median ROA']:+.2f})" for f in top10.index[:5]) + f". Strongest |ρ| = {top10.iloc[0]:.2f}. {n_sig} of {len(res)} features have p < 0.05, against about {0.05*len(res):.1f} expected by chance alone, so treat these as leads, not findings.",
 f"5. Direction: {n_neg} of the 10 strongest country-level associations are negative: listed firms in more developed, more open countries show slightly LOWER median ROA. Firm-year correlations point the same way but are much weaker (largest |ρ| = {res['ρ firm-year ROA'].abs().max():.2f}). Possible explanations (hypotheses, not tested): sector mix such as banks and utilities, asset-heavy business models, and faster-growing emerging markets.",
 f"6. Consistency: {int(chk['same sign in all 3'].sum())} of {len(chk)} features keep the same sign across the three views, but because the features move together this is mostly one signal.",
 f"7. EU vs non-EU: median of country medians {cn2[cn2.is_eu].median_roa.median():.3f} vs {cn2[~cn2.is_eu].median_roa.median():.3f} (Mann–Whitney p = {u.pvalue:.2f}, not corrected); no difference between EU groups (Kruskal–Wallis p = {kw.pvalue:.2f}).",
 f"8. Cross-checks: Yahoo vs ESEF country medians agree with ρ = {agree_r:.2f}; Yahoo median ROA vs Eurostat economy-wide corporate profitability: largest |ρ| = {eu_chk['ρ with Yahoo median ROA'].abs().max():.2f} (n≈21), i.e. no clear agreement.",
]
print("\n".join(lines))

1. Sample: 2,590 Yahoo companies in 34 countries (10,220 company-years). Every country has at least 10 companies.
2. Country features: the first principal component explains 34% of their variance (main drivers: regulatory_quality (-0.28), corruption_control (-0.28), rule_of_law (-0.28), gov_effectiveness (-0.27)); 15 of 1225 feature pairs have |ρ| ≥ 0.85, mostly among the governance indicators, GDP per person and the size/demography variables. So they overlap, but the 35 indicators are far from one single factor.
3. Variance in firm-year ROA, adjusted R²: company identity 0.635, country 0.041, sector 0.025, fiscal year 0.000; country + sector 0.067. Firm identity dominates; country conditions can explain only a few per cent.
4. Country level (n=34): the strongest associations with median ROA are internet_users (ρ=-0.43), trade_gdp (ρ=-0.40), life_expectancy (ρ=-0.39), urban_share (ρ=-0.35), pop_65plus (ρ=-0.33). Strongest |ρ| = 0.43. 4 of 35 features have p < 0.05, against about 1.8 ex

**Things to handle before modelling**
- Many country features are near-duplicates: use a few representatives (e.g. GDP per person, rule of law, a labour measure, R&D) or PCA components, not all 35.
- Firms in the same country share one feature value: a firm-year model needs country-clustered standard errors or country-level aggregation, otherwise significance is overstated.
- Sector must be controlled for; ESEF has no sector, so firm-level modelling uses the Yahoo sample (or ESEF with sectors added).
- Decide how to treat the same company appearing in both sources.
- The country sample is small (about 34): prefer simple, interpretable models, and report n and uncertainty everywhere.

In [23]:
con.close()